# Stage 2 v2 — ViT-B/16 focal loss
Chạy độc lập BCE control và focal $\gamma=0.5,1.0,1.5$. External test luôn bị khóa.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/HuyDuongQuoc/DATH-HK261-explainable-cxr.git"
REPO_BRANCH = "update_data"
REPO_DIR = Path("/content/DATH-HK261-explainable-cxr")
DRIVE_MOUNT = Path("/content/drive")
RUN_FINAL_TEST = False
MAX_NEW_RUNS_PER_SESSION = None
STAGE_ID = "stage2_focalloss_v2"
BOOTSTRAP_ROLE = "follower"
BOOTSTRAP_TIMEOUT_SECONDS = 600
STAGE2_VARIANTS = ["bce", "focal_g05", "focal_g10", "focal_g15"]
assert RUN_FINAL_TEST is False and len(set(STAGE2_VARIANTS)) == 4
STAGE_SPEC = {
    "notebook_id": "04_3_vit_b16_focalloss",
    "model_name": "vit_b_16",
    "model_slug": "vit_b_16",
    "config": "configs/vit_aug_auprc.yaml",
    "weighting": "sqrtcap3",
    "positive_weight": {"mode": "sqrt_ratio_capped", "max_value": 3.0},
    "stage_id": STAGE_ID,
    "bootstrap_role": BOOTSTRAP_ROLE,
    "bootstrap_timeout_seconds": BOOTSTRAP_TIMEOUT_SECONDS,
    "run_final_test": RUN_FINAL_TEST,
    "max_new_runs_per_session": MAX_NEW_RUNS_PER_SESSION,
    "export_path": "/content/exports/04_3_vit_b16_stage2_focalloss_v2_results.zip",
}
print(STAGE_SPEC)

In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT), force_remount=False)

In [ ]:
import base64, os, subprocess, sys
from getpass import getpass
github_token = getpass("GitHub PAT (leave blank for public repo): " ).strip()
git_env = os.environ.copy()
if github_token:
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}"})
def run_git(args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, env=git_env, check=True)
if REPO_DIR.exists():
    run_git(["fetch", "origin", REPO_BRANCH], REPO_DIR)
    run_git(["checkout", REPO_BRANCH], REPO_DIR)
    run_git(["pull", "--ff-only", "origin", REPO_BRANCH], REPO_DIR)
else:
    run_git(["clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip()
github_token = None
print("Repository commit:", commit_sha)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-common.txt")], check=True)

In [ ]:
from src.experiments.stage2_colab_v2 import build_stage2_experiments, prepare_stage2_colab, run_stage2_colab
experiments = build_stage2_experiments(STAGE_SPEC["model_name"], STAGE_SPEC["model_slug"], STAGE_SPEC["config"], STAGE_SPEC["weighting"], STAGE_SPEC["positive_weight"])
assert len(experiments) == 4 and len({item["run_name"] for item in experiments}) == 4
print([item["run_name"] for item in experiments])
STAGE_CONTEXT = prepare_stage2_colab(STAGE_SPEC, REPO_DIR, commit_sha)
print("STAGE 2 V2 FOLLOWER VERIFIED")

In [ ]:
analysis = run_stage2_colab(STAGE_SPEC, REPO_DIR, commit_sha, prepared_context=STAGE_CONTEXT)
display(analysis["macro"]); display(analysis["ranking"])